# Optimization and Numerical Integration (Python)

**Author:** Dimitris Rizopoulos

This notebook is a Python translation of the R vignette *Optimization and
Numerical Integration*, using the `glmmadaptive` Python package.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.special import expit

from glmmadaptive import MixedModel
from glmmadaptive.families import Binomial

---

## 1  Estimation Procedure

As described in the GLMMadaptive basics vignette, the log-likelihood function
behind the mixed models fitted by **glmmadaptive** contains an intractable
integral over the random effects.  Maximum likelihood estimation therefore
requires a combination of **numerical integration** and **optimization**.

### 1.1  Optimization

Two interlinked algorithms are implemented:

1. **EM algorithm** — treats the random effects as 'missing data'.  Generally
   more stable when starting far from the mode, but converges at a linear rate.
2. **Quasi-Newton algorithm** — direct maximization with super-linear
   convergence, but sensitive to starting values.

By default, the procedure starts with a fixed number of EM iterations to
refine the initial values, then switches to the quasi-Newton algorithm.

### 1.2  Numerical Integration

Integration over the random effects uses the **adaptive Gauss-Hermite
quadrature** rule, which requires locating the posterior modes of the random
effects for each sample unit.  To reduce computation, modes are not relocated
at every iteration, but every few iterations (`update_gh_every`).

### 1.3  Control Parameters

The `control` dict passed to `MixedModel` maps directly to the `control`
argument of R's `mixed_model()`.  The full correspondence is:

| R argument | Python `control` key | Default | Description |
|---|---|---|---|
| `nAGQ` | `n_agh` | 11 (≤2 REs), 7 otherwise | Number of quadrature points |
| `iter_EM` | `iter_em` | `30` | First-phase EM iterations |
| `update_GH_every` | `update_gh_every` | `10` | Modes update frequency (EM phase) |
| `optimizer` | `optimizer` | `"BFGS"` | `"BFGS"`, `"L-BFGS-B"`, `"Nelder-Mead"` |
| `iter_qN_outer` | `iter_qn_outer` | `15` | Outer quasi-Newton iterations |
| `iter_qN` | `iter_qn` | `10` | Inner optimizer iterations per outer step |
| `iter_qN_incr` | `iter_qn_incr` | `10` | Increment to `iter_qn` per outer step |

R's `optimizer = "nlminb"` has no direct equivalent; use `"L-BFGS-B"` as the
nearest substitute.

---

## 2  Controlling the Optimization and Integration

We start by simulating some data for a binary longitudinal outcome with a
categorical time variable:

In [ ]:
np.random.seed(1234)
n = 300   # number of subjects
K = 4     # number of measurements per subject
t_max = 15

# Categorical time variable: Time1, Time2, Time3, Time4
time_labels = np.tile([f"Time{k}" for k in range(1, K + 1)], n)
ids = np.repeat(np.arange(1, n + 1), K)
sex_labels = np.repeat(["male"] * (n // 2) + ["female"] * (n // 2), K)

DF = pd.DataFrame({"id": ids, "time": time_labels, "sex": sex_labels})

# Build design matrices matching R's model.matrix(~ sex * time)
time_dummies = pd.get_dummies(DF["time"], drop_first=True)  # Time1 is reference
sex_female   = (DF["sex"] == "female").astype(float)
interactions = time_dummies.multiply(sex_female, axis=0)
interactions.columns = [f"sex_female:{c}" for c in interactions.columns]

X = np.column_stack([
    np.ones(n * K),
    sex_female,
    time_dummies.values,
    interactions.values,
])
Z = np.ones((n * K, 1))

betas = np.array([-2.13, 1.0] + [1.2, -1.2] * (K - 1))
D11 = 1.0

b = np.random.normal(0, np.sqrt(D11), n)
eta_y = X @ betas + b[DF["id"].values - 1]
DF["y"] = np.random.binomial(1, expit(eta_y))

DF.head(8)

### 2.1  Default control

Fit with default control arguments
(`iter_em=30`, `iter_qn_outer=15`, `n_agh=11`, `optimizer="BFGS"`):

In [ ]:
fm_default = MixedModel(
    fixed="y ~ sex + time",
    random="~ 1 | id",
    family=Binomial(),
    data=DF,
).fit()

print(fm_default.summary())

### 2.2  Skip EM, use L-BFGS-B

Skip the EM phase entirely (`iter_em=0`), use `"L-BFGS-B"` (nearest
equivalent to R's `nlminb`), and increase the inner iteration budget more
aggressively with `iter_qn_incr=5`:

In [ ]:
fm_no_em = MixedModel(
    fixed="y ~ sex + time",
    random="~ 1 | id",
    family=Binomial(),
    data=DF,
    control={"iter_em": 0, "optimizer": "L-BFGS-B", "iter_qn_incr": 5},
).fit()

print(fm_no_em.summary())

### 2.3  EM-only convergence

Allow up to 1000 EM iterations, update quadrature modes every 5 EM steps, and
use 21 quadrature points:

In [ ]:
fm_only_em = MixedModel(
    fixed="y ~ sex + time",
    random="~ 1 | id",
    family=Binomial(),
    data=DF,
    control={"iter_em": 1000, "update_gh_every": 5, "n_agh": 21},
).fit()

print(fm_only_em.summary())

### 2.4  No optimization at all

Set both `iter_em=0` and `iter_qn_outer=0` to skip all optimization and return
the model evaluated at its initial values (useful for diagnostics):

In [ ]:
fm_no_opt = MixedModel(
    fixed="y ~ sex + time",
    random="~ 1 | id",
    family=Binomial(),
    data=DF,
    control={"iter_em": 0, "iter_qn_outer": 0},
).fit()

print("logLik at initial values:", fm_no_opt.logLik)

---

## 3  Advice on Optimization and Numerical Integration Control

**EM overshoot.**  Even though the EM algorithm is generally more stable than
the quasi-Newton, it can sometimes overshoot and fail to bring the parameters
near the (local) maximum.  In these cases set `iter_em=0` to use only the
quasi-Newton phase.

**Initial values.**  Convergence problems may sometimes be attributed to poor
initial values.  The `initial_values` argument of `MixedModel` can be used to
override the default initialization.  For example, for a random intercepts and
random slopes model:

In [ ]:
# Example: manually supply initial values (model not run here, just shows the syntax)
n_fixed_effects = 2  # intercept + sex

_ = MixedModel(
    fixed="y ~ sex",
    random="~ time | id",
    family=Binomial(),
    data=DF,
    initial_values={
        "betas": np.zeros(n_fixed_effects),
        "D": np.array([[0.5, 0.0], [0.0, 0.1]]),
    },
)
print("MixedModel with custom initial values constructed successfully.")

**Coefficient scaling.**  Coefficients that differ by several orders of
magnitude can lead to convergence issues.  Scale variables so that the
coefficients are on the same magnitude.

**Separation.**  For dichotomous and count data, (complete) separation issues
may be encountered.  R's `mixed_model()` has a `penalized` argument that places
a Student's t penalty (mean 0, scale 1, df 3) on the fixed effects coefficients.
> **Note:** The `penalized` argument is not yet implemented in the Python port.

**Quadrature points.**  Increase `n_agh` until the coefficients and
log-likelihood value stabilise.  However, too many quadrature points can cause
overflow.  In the majority of cases, 11–15 quadrature points suffice.